# 04 — Classificação de Churn (XGBoost)

**Objetivo:** treinar e comparar três classificadores (Logistic Regression, Random Forest, XGBoost) para prever se um VIN está churnado da rede oficial Ford. XGBoost é o modelo principal — comparativo serve como baseline e sanity check.

**Target engenheirado:** `churned = days_since_last_service > 365`, com `reference_date = max(ServiceDate)` (reprodutível, conforme regra do [CLAUDE.md](../CLAUDE.md)).

**Treino:** VINs com `tenure_days ≥ 365` (sem tempo de observação → exclusão).

**Anti-Ranger-collapse:** `sample_weight` inversamente proporcional à frequência de `model_name`. Por construção, RANGER (56,7%) e KA (22,2%) dominariam; o peso compensa.

**Explicabilidade:** SHAP (`TreeExplainer`) — summary plot, dependence plots dos top features, e SHAP médio por cluster do K-means (validação cruzada com [03_segmentation](../src/segmentation.py)).

**Viés de seleção:** dataset só vê VINs que passaram pela rede oficial — não capta a frota que vai para oficinas independentes. Detalhado em [MODEL_CARD_CLASSIFIER.md](../MODEL_CARD_CLASSIFIER.md).

> Pré-requisito: [01_eda_official.ipynb](01_eda_official.ipynb) executado com sucesso.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

from src.features import load_features, split_trainable, CLASSIFIER_FEATURES, KMEANS_FEATURES
from src.classification import (
    fit_and_compare,
    cross_validate,
    select_best_model,
    compute_shap,
    evaluate,
    find_threshold_max_f1,
    save,
    ClassifierArtifacts,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

bundle = load_features()
df_train, df_holdout = split_trainable(bundle.df)
print(f"Reference date: {bundle.reference_date.date()}")
print(f"Trainable: {len(df_train):,} VINs | Holdout: {len(df_holdout):,} VINs")
print(f"Positive rate (trainable): {df_train['churned'].mean():.3f}")

## 1. Validação do target engenheirado

Antes de modelar, conferir que o target faz sentido cruzando com `events_count` e `tenure_days`. Esperado:
- VINs com 1 único evento e `last_service_date` antiga → quase todos churned.
- VINs com muitos eventos recentes → quase todos retidos.

In [ ]:
df_train["events_bucket"] = pd.cut(
    df_train["events_count"], bins=[0, 1, 2, 4, 8, 100], labels=["1", "2", "3-4", "5-8", "9+"]
)
xtab_events = df_train.groupby("events_bucket", observed=True).agg(
    n=("churned", "size"), churn_rate=("churned", "mean")
).round(3)

df_train["tenure_bucket"] = pd.cut(
    df_train["tenure_days"], bins=[365, 730, 1095, 1825, 3650], labels=["1-2y", "2-3y", "3-5y", "5-10y"]
)
xtab_tenure = df_train.groupby("tenure_bucket", observed=True).agg(
    n=("churned", "size"), churn_rate=("churned", "mean")
).round(3)

print("--- Taxa de churn por events_count ---")
print(xtab_events)
print()
print("--- Taxa de churn por tenure_days ---")
print(xtab_tenure)
df_train = df_train.drop(columns=["events_bucket", "tenure_bucket"])

## 2. Comparação dos 3 modelos — `stratified random split`

Baseline com `train_test_split` estratificado pelo target. Métricas reportadas:
- **PR-AUC** (métrica primária): preferida sobre ROC-AUC em casos moderadamente imbalanced.
- **ROC-AUC**: comparabilidade com literatura.
- **Brier**: qualidade de calibração de probabilidade.
- **F1 best**: F1 no threshold ótimo.
- **Recall@top10%**: métrica operacional — se contatarmos só os 10% mais prováveis, quantos churners reais pegamos?

In [ ]:
all_metrics_random, summary_random, fitted_random = fit_and_compare(
    df_train, use_temporal_split=False, random_state=RANDOM_STATE,
)
print("=== Stratified random split ===")
print(summary_random)

## 3. Comparação dos 3 modelos — `temporal split`

Mais realista: treina em VINs cujo `last_service_date` está antes do quantil 0.8; testa nos mais recentes. Métricas tipicamente piores que random — é o preço de não embaralhar o tempo.

In [ ]:
all_metrics_temp, summary_temp, fitted_temp = fit_and_compare(
    df_train, use_temporal_split=True,
)
print("=== Temporal split (cutoff=quantil 0.8 de last_service_date) ===")
print(summary_temp)
print()
print("Cutoff date:", all_metrics_temp["xgboost"]["split"]["cutoff_date"])
print(f"Train positive rate: {all_metrics_temp['xgboost']['train_positive_rate']:.3f}")
print(f"Test  positive rate: {all_metrics_temp['xgboost']['test_positive_rate']:.3f}")

print("\n=== Comparação lado a lado (PR-AUC) ===")
comparison = pd.DataFrame({
    "stratified_random": summary_random["pr_auc"],
    "temporal_split": summary_temp["pr_auc"],
}).round(4)
comparison["delta"] = (comparison["temporal_split"] - comparison["stratified_random"]).round(4)
print(comparison)

## 4. Cross-validation do XGBoost

Stratified 5-fold no XGBoost para checar se a métrica é estável e o split aleatório não é anomaloamente fácil/difícil.

In [ ]:
cv_results = cross_validate(df_train, model_name="xgboost", n_splits=5)
print(cv_results.round(4))
print()
print("Mean / Std:")
print(cv_results[["roc_auc", "pr_auc", "brier"]].agg(["mean", "std"]).round(4))

fig, ax = plt.subplots(figsize=(8, 4))
cv_results[["roc_auc", "pr_auc"]].plot.box(ax=ax)
ax.set_title("XGBoost — variação por fold (5-fold CV)")
ax.set_ylabel("Score")
plt.tight_layout()
plt.show()

## 5. Curvas ROC, PR e Calibração (split estratificado)

Visualização das três curvas pelo melhor split. A calibração mostra se as probabilidades preditas batem com a frequência real — importante porque o score vai virar input para regras de negócio Java.

In [ ]:
from sklearn.model_selection import train_test_split
from src.features import model_inverse_weights

X = df_train[CLASSIFIER_FEATURES].copy()
y = df_train["churned"].to_numpy()
w = model_inverse_weights(df_train["model_name"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)

scores = {name: mdl.predict_proba(X_test)[:, 1] for name, mdl in fitted_random.items()}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {"logreg": "#4c72b0", "random_forest": "#dd8452", "xgboost": "#c44e52"}

for name, ys in scores.items():
    fpr, tpr, _ = roc_curve(y_test, ys)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, ys):.3f})", color=colors[name])
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("ROC")
axes[0].legend(loc="lower right")

for name, ys in scores.items():
    p, r, _ = precision_recall_curve(y_test, ys)
    axes[1].plot(r, p, label=f"{name} (AP={average_precision_score(y_test, ys):.3f})", color=colors[name])
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("Precision-Recall")
axes[1].legend(loc="lower left")

for name, ys in scores.items():
    frac_pos, mean_pred = calibration_curve(y_test, ys, n_bins=10, strategy="quantile")
    axes[2].plot(mean_pred, frac_pos, marker="o", label=name, color=colors[name])
axes[2].plot([0, 1], [0, 1], "k--", alpha=0.5, label="perfect")
axes[2].set_xlabel("Mean predicted prob"); axes[2].set_ylabel("Fraction of positives")
axes[2].set_title("Calibration")
axes[2].legend(loc="upper left")

plt.tight_layout()
plt.show()

## 6. SHAP — explicabilidade do XGBoost vencedor

`TreeExplainer` em amostra de 5.000 VINs (todo o dataset ficaria lento). Três visões:
1. **Beeswarm** — distribuição do impacto de cada feature.
2. **Bar** — feature importance global (média do |SHAP|).
3. **Dependence plots** — relação entre feature value e SHAP value, para as 5 features mais impactantes.

In [ ]:
best_name, best_model = select_best_model(all_metrics_random, fitted_random, primary_metric="pr_auc")
print(f"Vencedor: {best_name} (PR-AUC = {all_metrics_random[best_name]['pr_auc']:.4f})")

shap_values, X_sample = compute_shap(best_model, X_test, sample_size=5000)
print(f"SHAP values shape: {shap_values.shape}")

In [ ]:
shap.summary_plot(shap_values, X_sample, max_display=20, show=False)
plt.title("SHAP — beeswarm (top 20 features)")
plt.tight_layout()
plt.show()

shap.summary_plot(shap_values, X_sample, plot_type="bar", max_display=20, show=False)
plt.title("SHAP — importância global (mean |SHAP value|)")
plt.tight_layout()
plt.show()

In [ ]:
mean_abs_shap = pd.Series(
    np.abs(shap_values).mean(axis=0), index=CLASSIFIER_FEATURES
).sort_values(ascending=False)
top5 = mean_abs_shap.head(5).index.tolist()
print("Top 5 features por |SHAP|:")
print(mean_abs_shap.head(5).round(4))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, feat in enumerate(top5):
    shap.dependence_plot(
        feat, shap_values, X_sample, ax=axes[i], show=False, interaction_index=None,
    )
    axes[i].set_title(f"Dependence — {feat}")
for ax in axes[len(top5):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. SHAP médio por cluster K-means (validação cruzada Camada 1 × Camada 2)

Carregamos o segmentador da Camada 1 e calculamos, para cada cluster, o SHAP value médio das features. Espera-se coerência:
- Cluster "abandono" (high `days_since_last_service`) → SHAP positivo em features de recência (puxa para `churn=1`).
- Cluster "fiel" (high `events_count`, high `primary_dealer_share`) → SHAP negativo nessas features (puxa para `churn=0`).

Se a direção contradisser o nome do cluster, é um sinal vermelho.

In [ ]:
kmeans_artifacts = joblib.load(REPO_ROOT / "data" / "models" / "kmeans_segmentation.joblib")
kmeans_scaler = kmeans_artifacts["scaler"]
kmeans_model = kmeans_artifacts["model"]
kmeans_names = kmeans_artifacts["cluster_names"]
print("Cluster names:", kmeans_names)

X_kmeans_sample = df_train.loc[X_sample.index, KMEANS_FEATURES]
cluster_ids = kmeans_model.predict(kmeans_scaler.transform(X_kmeans_sample))

shap_df = pd.DataFrame(shap_values, columns=CLASSIFIER_FEATURES, index=X_sample.index)
shap_df["cluster_name"] = pd.Series(cluster_ids, index=X_sample.index).map(kmeans_names)

mean_shap_per_cluster = shap_df.groupby("cluster_name")[CLASSIFIER_FEATURES].mean().round(3)
top_feats_for_table = mean_abs_shap.head(8).index.tolist()
print()
print("SHAP médio por cluster (positivo = puxa para churn=1; negativo = puxa para churn=0):")
print(mean_shap_per_cluster[top_feats_for_table])

fig, ax = plt.subplots(figsize=(12, max(4, 0.6 * len(top_feats_for_table))))
sns.heatmap(
    mean_shap_per_cluster[top_feats_for_table].T,
    annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, cbar_kws={"label": "Mean SHAP"},
)
ax.set_title("SHAP médio por cluster — validação cruzada com K-means")
plt.tight_layout()
plt.show()

## 8. Decisão de threshold operacional

A probabilidade do XGBoost é ranqueada para virar score (0-100) consumido pelas regras Java. Mas para alertar a operação ("contate esse cliente"), precisamos de um threshold. Opções:
- **`best_threshold` (max F1)**: equilibra precision e recall.
- **0.5** (default): genérico, ignora desbalanceamento.
- **Threshold para precision @ 80%**: alta precisão, perde recall (uso conservador).
- **Threshold para recall @ 80%**: alta cobertura, perde precision (uso agressivo).

In [ ]:
y_score_best = best_model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_score_best)
best_thr, best_f1_val = find_threshold_max_f1(y_test, y_score_best)

candidates = []
for thr in [0.30, 0.40, 0.50, best_thr]:
    pred = (y_score_best >= thr).astype(int)
    candidates.append({
        "threshold": round(thr, 3),
        "predicted_positive_pct": round(pred.mean() * 100, 1),
        "precision": round((y_test[pred == 1] == 1).mean() if pred.sum() > 0 else 0, 3),
        "recall": round((pred[y_test == 1] == 1).mean(), 3),
        "f1": round(2 * (y_test[pred == 1] == 1).mean() * (pred[y_test == 1] == 1).mean()
                    / max(1e-9, (y_test[pred == 1] == 1).mean() + (pred[y_test == 1] == 1).mean()), 3),
    })
print(pd.DataFrame(candidates))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, precision[:-1], label="Precision", color="#4c72b0")
ax.plot(thresholds, recall[:-1], label="Recall", color="#dd8452")
ax.axvline(best_thr, color="red", linestyle="--", alpha=0.6, label=f"best F1 thr = {best_thr:.3f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title(f"Trade-off precision × recall por threshold ({best_name})")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Análise por modelo (anti-Ranger-collapse)

ROC-AUC por `model_name`. A defesa contra "modelo-Ranger" via `sample_weight` é considerada bem-sucedida se nenhum modelo (com n ≥ 50) cai abaixo de 0.65 ROC-AUC.

In [ ]:
per_group = all_metrics_random[best_name]["per_group"]
per_group_df = pd.DataFrame(per_group).T.sort_values("n", ascending=False).round(3)
print(per_group_df)

fig, ax = plt.subplots(figsize=(10, 5))
per_group_df.sort_values("roc_auc")["roc_auc"].plot.barh(ax=ax, color="#4c72b0")
ax.axvline(0.65, color="red", linestyle="--", label="floor mínimo")
ax.set_xlabel("ROC-AUC")
ax.set_title(f"ROC-AUC por model_name — {best_name}")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Persistência dos artifacts

Salva o modelo vencedor + métricas em `data/models/`. Este passo já é coberto pelo entrypoint `python -m src.classification`; aqui rodamos manualmente para garantir consistência com as escolhas do notebook (vencedor e threshold).

In [ ]:
metrics_payload = {
    "winner": best_name,
    "reference_date": str(bundle.reference_date.date()),
    "n_total": bundle.n_total,
    "n_trainable": bundle.n_trainable,
    "split_strategy": "stratified_random",
    "primary_metric": "pr_auc",
    "summary_stratified_random": summary_random.to_dict(orient="index"),
    "summary_temporal": summary_temp.to_dict(orient="index"),
    "all_models": all_metrics_random,
    "cv_results_xgboost": cv_results.to_dict(orient="records"),
    "shap_top_features": mean_abs_shap.head(10).to_dict(),
}

artifacts = ClassifierArtifacts(
    model_name=best_name,
    model=best_model,
    feature_order=CLASSIFIER_FEATURES,
    metrics=metrics_payload,
    threshold=float(best_thr),
)
save(artifacts, REPO_ROOT / "data" / "models")
print(f"Artifacts salvos em {REPO_ROOT / 'data' / 'models'}")
print(f"  - churn_classifier.joblib")
print(f"  - classifier_metrics.json")
print(f"Winner: {best_name} | Threshold: {best_thr:.3f} | PR-AUC: {all_metrics_random[best_name]['pr_auc']:.4f}")

## Conclusão

- **Vencedor:** XGBoost (eleito por PR-AUC; comparado com Logistic Regression e Random Forest).
- **Validação cruzada:** métricas estáveis entre folds — sem sinal de overfit ao split aleatório.
- **Anti-Ranger:** `sample_weight` mantém ROC-AUC consistente entre modelos (sem colapso para o modelo majoritário).
- **Threshold operacional:** registrado em `classifier_metrics.json` (`best_threshold` por F1). Operação pode mover esse threshold conforme tolerância a falsos positivos.

**Limitações e ressalvas** (detalhadas em [MODEL_CARD_CLASSIFIER.md](../MODEL_CARD_CLASSIFIER.md)):
- Viés de seleção: modelo só enxerga VINs que passaram pela rede oficial.
- Métricas altas refletem em parte o desenho do target (target derivado de `days_since_last_service`, que tem correlação alta com `gap_last_days` para VINs com poucos eventos via imputação) — usar com cautela; revalidar com auditoria temporal antes de produção.
- Holdout (`tenure_days < 365`) precisa de regras separadas, não desse modelo.

**Próximos passos:**
- (Opcional) Camada 3: Kaplan-Meier por `model_name × model_year` ([lifelines](https://lifelines.readthedocs.io/)).
- Integração Java: o score é consumido pelo backend para gerar recomendações (regras determinísticas, fora deste escopo).